In [0]:
# Create (idempotently) the Databricks Lakehouse inference monitor on the inference log.
# Requires workspace support for quality monitors. Safe to re-run: skips when a monitor already exists.
import sys
import os
sys.path.insert(0, os.getcwd())

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorInferenceLog, MonitorInferenceLogProblemType
from utils import load_project, table

cfg = load_project()
dbutils.widgets.text("catalog", "stock_demo")
catalog = dbutils.widgets.get("catalog")
w = WorkspaceClient()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.monitoring")
log_table = table(cfg, "inference_log", catalog)
if not spark.catalog.tableExists(log_table):
    raise RuntimeError(f"{log_table} does not exist. Run batch_inference.py (or simulate_drift.py) first.")

try:
    w.quality_monitors.get(table_name=log_table)
    print(f"Monitor already exists on {log_table} - nothing to do.")
except Exception:
    w.quality_monitors.create(
        table_name=log_table,
        assets_dir=f"/Shared/{catalog}/stock_monitoring",
        output_schema_name=f"{catalog}.monitoring",
        inference_log=MonitorInferenceLog(
            problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_REGRESSION,
            prediction_col="prediction",
            label_col="actual_next_close",
            model_id_col="model_version",
            timestamp_col="request_ts",
            granularities=["1 day"],
        ),
    )
    print("Inference monitor created.")